# CocinaAI — EDA e Ingeniería de Features

**Proyecto:** CocinaAI — Tu chef de alacena

**Fuentes:** Spoonacular API (recetas mexicanas) + TheMealDB (área Mexican)

**Objetivo:** Analizar patrones de ingredientes en cocina mexicana para respaldar el diseño del sistema de recomendación.

### Imports

In [31]:
!pip install requests pandas numpy plotly scikit-learn -q

In [32]:
import json
import time
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import scipy.sparse
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [33]:
from google.colab import userdata
spoonacular_api = userdata.get('SPOONACULAR_API')
print('API key cargada:', 'OK' if spoonacular_api else 'ERROR - revisa Secrets')

API key cargada: OK


### Variables globales

In [34]:
N_SPOONACULAR = 30  # recetas de Spoonacular
# TheMealDB: solo area Mexican (autenticas), sin categorias genericas
THEMEALDB_AREAS = ['Mexican']

## 1. Funciones de ingesta

In [35]:
def get_spoonacular_recipes(cuisine, number, api_key):
    """Obtiene recetas mexicanas desde Spoonacular."""
    url = 'https://api.spoonacular.com/recipes/complexSearch'
    params = {
        'cuisine': cuisine, 'number': number,
        'addRecipeInformation': True, 'addRecipeNutrition': True,
        'apiKey': api_key
    }
    resp = requests.get(url, params=params, timeout=15)
    return resp.json().get('results', [])


In [79]:
def parse_spoonacular(recipe):
    """Normaliza receta de Spoonacular."""

    nutrition = recipe.get('nutrition', {})
    ingredients = nutrition.get('ingredients', [])
    names = [i.get('name','').lower().strip() for i in ingredients if i.get('name')]

    return {
        'id': f"sp_{recipe.get('id')}",
        'titulo': recipe.get('title',''),
        'fuente': 'Spoonacular',
        'tiempo_minutos': recipe.get('readyInMinutes'),
        'porciones': recipe.get('servings'),
        'num_ingredientes': len(names),
        'ingredientes': names,
        'vegana': recipe.get('vegan', False),
        'vegetariana': recipe.get('vegetarian', False),
        'sin_gluten': recipe.get('glutenFree', False),
        'url': recipe.get('sourceUrl',''),
        'imagen': recipe.get('image',''),
    }


In [80]:
def get_themealdb_by_area(area):
    """Obtiene recetas de TheMealDB por area geografica."""
    url = f'https://www.themealdb.com/api/json/v1/1/filter.php?a={area}'
    resp = requests.get(url, timeout=15)
    return resp.json().get('meals', []) or []

def get_themealdb_detail(meal_id):
    """Obtiene detalle completo de una receta de TheMealDB."""
    url = f'https://www.themealdb.com/api/json/v1/1/lookup.php?i={meal_id}'
    resp = requests.get(url, timeout=15)
    meals = resp.json().get('meals', [])
    return meals[0] if meals else {}


In [81]:
def parse_themealdb(meal):
    """Normaliza receta de TheMealDB."""
    ingredients = []
    for i in range(1, 21):
        ing = meal.get(f'strIngredient{i}', '')
        if ing and ing.strip():
            ingredients.append(ing.lower().strip())
    category = meal.get('strCategory','').lower()
    tags = (meal.get('strTags') or '').lower()
    is_veg = any(v in category or v in tags for v in ['vegetarian','vegan','veggie'])
    return {
        'id': f"mdb_{meal.get('idMeal')}",
        'titulo': meal.get('strMeal',''),
        'fuente': 'TheMealDB',
        'tiempo_minutos': None,
        'porciones': None,
        'num_ingredientes': len(ingredients),
        'ingredientes': ingredients,
        'vegana': 'vegan' in category,
        'vegetariana': is_veg,
        'sin_gluten': False,
        'url': meal.get('strSource',''),
        'imagen': meal.get('strMealThumb',''),
    }


## 2. Descarga de datos

In [82]:
# Spoonacular
print('Descargando Spoonacular...')
raw_sp = get_spoonacular_recipes('mexican', N_SPOONACULAR, spoonacular_api)
sp_list = [parse_spoonacular(r) for r in raw_sp]
print(f'  Spoonacular: {len(sp_list)} recetas')

Descargando Spoonacular...
  Spoonacular: 100 recetas


In [83]:
# TheMealDB — solo area Mexican
print('Descargando TheMealDB (area Mexican)...')
mdb_ids = set()
mdb_list = []

for area in THEMEALDB_AREAS:
    for m in get_themealdb_by_area(area):
        if m['idMeal'] not in mdb_ids:
            mdb_ids.add(m['idMeal'])
            detail = get_themealdb_detail(m['idMeal'])
            if detail:
                mdb_list.append(parse_themealdb(detail))
            time.sleep(0.1)

print(f'  TheMealDB: {len(mdb_list)} recetas')

Descargando TheMealDB (area Mexican)...
  TheMealDB: 6 recetas


In [84]:
# Fusion — Spoonacular primero para ganar en deduplicacion
df_sp  = pd.DataFrame(sp_list)  if sp_list  else pd.DataFrame()
df_mdb = pd.DataFrame(mdb_list) if mdb_list else pd.DataFrame()

recipes_df = pd.concat([df_sp, df_mdb], ignore_index=True)
print(f'Antes de limpiar: {len(recipes_df)} recetas')
print(recipes_df['fuente'].value_counts())

Antes de limpiar: 106 recetas
fuente
Spoonacular    100
TheMealDB        6
Name: count, dtype: int64


In [85]:
print(f'df_sp shape: {df_sp.shape}')
print(f'df_mdb shape: {df_mdb.shape}')
print()
print('Primeras filas de df_sp:')
print(df_sp[['titulo','fuente']].head(3))

df_sp shape: (100, 12)
df_mdb shape: (6, 12)

Primeras filas de df_sp:
                                      titulo       fuente
0                         Corn Avocado Salsa  Spoonacular
1  Cheesy Chicken Enchilada Quinoa Casserole  Spoonacular
2                         Homemade Guacamole  Spoonacular


## 3. Limpieza

In [86]:
n = len(recipes_df)

# 1. Sin ingredientes
recipes_df = recipes_df[recipes_df['num_ingredientes'] > 0]
print(f'Eliminadas sin ingredientes: {n - len(recipes_df)}')

# 2. Duplicados por titulo (Spoonacular gana al ir primero)
n = len(recipes_df)
recipes_df = recipes_df.drop_duplicates(subset='titulo', keep='first')
print(f'Eliminadas por titulo duplicado: {n - len(recipes_df)}')

# 3. Outliers de tiempo — solo eliminar si tiempo es muy extremo
n = len(recipes_df)
recipes_df = recipes_df[
    recipes_df['tiempo_minutos'].isna() |
    (recipes_df['tiempo_minutos'] <= 480)
]
print(f'Eliminadas por tiempo > 480 min: {n - len(recipes_df)}')

# 4. Imputar tiempo faltante (TheMealDB no tiene tiempo)
tiempo_mediana = recipes_df['tiempo_minutos'].median()
recipes_df['tiempo_minutos'] = recipes_df['tiempo_minutos'].fillna(tiempo_mediana)
recipes_df['porciones'] = recipes_df['porciones'].fillna(4)

recipes_df = recipes_df.reset_index(drop=True)
print(f'\nRecetas limpias: {len(recipes_df)}')
print(recipes_df['fuente'].value_counts())

Eliminadas sin ingredientes: 0
Eliminadas por titulo duplicado: 1
Eliminadas por tiempo > 480 min: 2

Recetas limpias: 103
fuente
Spoonacular    98
TheMealDB       5
Name: count, dtype: int64


## 4. Ingeniería de variables

### 4.1 Dificultad

In [87]:
def clasificar_dificultad(num_ing, tiempo):
    score = 0
    if num_ing > 12: score += 2
    elif num_ing > 7: score += 1
    if tiempo > 60: score += 2
    elif tiempo > 30: score += 1
    return 'Facil' if score <= 1 else ('Media' if score <= 3 else 'Dificil')

recipes_df['dificultad'] = recipes_df.apply(
    lambda r: clasificar_dificultad(r['num_ingredientes'], r['tiempo_minutos']), axis=1
)
recipes_df['dificultad'].value_counts()

,count
dificultad,
Media,83
Facil,19
Dificil,1


### 4.2 Score de alacena

In [88]:
# Top 60 ingredientes mas frecuentes del dataset
all_ings = [i for sublist in recipes_df['ingredientes'] for i in sublist]
ingredient_counts = Counter(all_ings)
top_60 = set([ing for ing, _ in ingredient_counts.most_common(60)])

print('Top 20 ingredientes mas frecuentes:')
print([ing for ing, _ in ingredient_counts.most_common(20)])

Top 20 ingredientes mas frecuentes:
['onion', 'cilantro', 'garlic', 'olive oil', 'salt', 'juice of lime', 'chili powder', 'bell pepper', 'pepper', 'black beans', 'avocado', 'cumin', 'corn tortillas', 'cream', 'salt and pepper', 'tomatoes', 'ground cumin', 'paprika', 'lime juice', 'salsa']


In [89]:
# Proporcion de ingredientes comunes de alacena
recipes_df['prop_alacena'] = recipes_df['ingredientes'].apply(
    lambda ings: round(len(set(ings) & top_60) / max(len(ings), 1), 3)
)

# Pocos ingredientes (mas facil de tener todo)
recipes_df['pocos_ingredientes'] = (recipes_df['num_ingredientes'] <= 7).astype(int)

# Receta rapida
recipes_df['es_rapida'] = (recipes_df['tiempo_minutos'] <= 35).astype(int)

# Score compuesto
recipes_df['score_alacena'] = (
    0.60 * recipes_df['prop_alacena'] +
    0.25 * recipes_df['pocos_ingredientes'] +
    0.15 * recipes_df['es_rapida']
).round(3)

print('Score alacena:')
print(recipes_df['score_alacena'].describe().round(3))

Score alacena:
count   103.00
mean      0.42
std       0.16
min       0.13
25%       0.30
50%       0.40
75%       0.48
max       0.90
Name: score_alacena, dtype: float64


In [90]:
# Top 10 recetas mas aprovechables
recipes_df[['titulo','fuente','num_ingredientes','score_alacena','dificultad']]\
    .sort_values('score_alacena', ascending=False).head(10)

,titulo,fuente,num_ingredientes,score_alacena,dificultad
0,Corn Avocado Salsa,Spoonacular,6,0.90,Facil
47,Instant Pot Chicken Tacos,Spoonacular,6,0.90,Facil
2,Homemade Guacamole,Spoonacular,6,0.85,Facil
64,How to Make a Chicken Taco Crock Pot,Spoonacular,6,0.85,Media
12,Knock-Your-Socks-Off Guacamole,Spoonacular,7,0.83,Facil
94,Enchilada Chicken,Spoonacular,6,0.75,Facil
28,Mango Kiwi Salsa,Spoonacular,10,0.69,Facil
48,"Mushroom, Jalapeño, and Cilantro Salsa",Spoonacular,7,0.68,Facil
3,Instant Pot Chicken Taco Soup,Spoonacular,12,0.65,Facil
50,Mango Guacamole,Spoonacular,6,0.65,Facil


### 4.3 Normalizacion

In [91]:
cols_norm = ['num_ingredientes','tiempo_minutos','score_alacena','prop_alacena']
scaler = MinMaxScaler()
norm_vals = scaler.fit_transform(recipes_df[cols_norm].fillna(0))
norm_df = pd.DataFrame(norm_vals, columns=[f'{c}_norm' for c in cols_norm], index=recipes_df.index)
recipes_df = pd.concat([recipes_df, norm_df], axis=1)
print('Normalizacion completa.')

Normalizacion completa.


### 4.4 TF-IDF de ingredientes

In [92]:
# Texto de ingredientes para vectorizacion
recipes_df['ingredientes_texto'] = recipes_df['ingredientes'].apply(
    lambda ings: ' '.join([i.replace(' ','_') for i in ings if i])
)

print(f'Recetas para TF-IDF: {len(recipes_df)}')
print('Muestra:')
print(recipes_df['ingredientes_texto'].head(3).tolist())

Recetas para TF-IDF: 103
Muestra:
['avocado balsamic_vinegar cumin corn garlic bell_pepper', 'avocado pepper black_beans canned_tomatoes chili_powder quinoa cumin verde_enchilada_sauce cilantro green_onion_tops roma_tomato salt cheese chicken_breast corn pepper', 'avocados cilantro juice_of_lime pepper onion roma_tomato']


In [93]:
# Ajustamos min_df segun el numero de recetas disponibles
n_recetas = len(recipes_df)
min_df = max(1, int(n_recetas * 0.02))  # al menos 2% de recetas
print(f'min_df calculado: {min_df} (para {n_recetas} recetas)')

tfidf = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    min_df=min_df
)
tfidf_matrix = tfidf.fit_transform(recipes_df['ingredientes_texto'])
print(f'Matriz TF-IDF: {tfidf_matrix.shape}')

min_df calculado: 2 (para 103 recetas)
Matriz TF-IDF: (103, 200)


In [94]:
# Ingredientes con mayor peso TF-IDF
feature_names = tfidf.get_feature_names_out()
avg_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_tfidf = pd.DataFrame({'ingrediente': feature_names, 'peso': avg_tfidf})\
            .sort_values('peso', ascending=False).head(20)
top_tfidf

,ingrediente,peso
143,onion,0.09
48,cilantro,0.07
139,olive_oil,0.07
179,salt,0.07
76,garlic,0.07
8,bell_pepper,0.06
112,juice_of_lime,0.06
13,black_beans,0.05
161,pepper,0.05
37,chili_powder,0.05


## 5. Análisis exploratorio

In [95]:
print('Dataset final:')
print(f'  Total recetas: {len(recipes_df)}')
print(f'  Por fuente:')
print(recipes_df['fuente'].value_counts())
print(f'\nEstadisticas:')
print(recipes_df[['tiempo_minutos','num_ingredientes','score_alacena']].describe().round(2))

Dataset final:
  Total recetas: 103
  Por fuente:
fuente
Spoonacular    98
TheMealDB       5
Name: count, dtype: int64

Estadisticas:
       tiempo_minutos  num_ingredientes  score_alacena
count          103.00            103.00         103.00
mean            47.43             13.13           0.42
std             34.80              5.05           0.16
min              5.00              6.00           0.13
25%             45.00             10.00           0.30
50%             45.00             12.00           0.40
75%             45.00             15.00           0.48
max            270.00             32.00           0.90


In [96]:
fig = px.pie(recipes_df, names='fuente',
    title='Recetas por fuente',
    color_discrete_map={'Spoonacular':'#B5722A','TheMealDB':'#1D9E75'},
    template='plotly_white')
fig.show()

In [97]:
# Top ingredientes
top_ing_df = pd.DataFrame(ingredient_counts.most_common(20), columns=['ingrediente','frecuencia'])
fig = go.Figure(go.Bar(x=top_ing_df['ingrediente'], y=top_ing_df['frecuencia'], marker_color='#B5722A'))
fig.update_layout(title='Top 20 ingredientes mas frecuentes', xaxis_tickangle=-45, template='plotly_white')
fig.show()

In [98]:
# Score alacena por dificultad
sc = recipes_df.groupby('dificultad')['score_alacena'].mean().round(3).reset_index()
fig = go.Figure(go.Bar(x=sc['dificultad'], y=sc['score_alacena'],
    marker_color=['#1D9E75','#EF9F27','#D85A30']))
fig.update_layout(title='Score de alacena promedio por dificultad', template='plotly_white')
fig.show()

In [99]:
# Scatter ingredientes vs score
fig = px.scatter(recipes_df, x='num_ingredientes', y='score_alacena',
    color='dificultad', hover_name='titulo',
    title='Numero de ingredientes vs Score de alacena',
    color_discrete_map={'Facil':'#1D9E75','Media':'#EF9F27','Dificil':'#D85A30'},
    template='plotly_white')
fig.show()

In [100]:
print('Hallazgo clave:')
for dif in ['Facil','Media','Dificil']:
    score = recipes_df[recipes_df['dificultad']==dif]['score_alacena'].mean()
    n = len(recipes_df[recipes_df['dificultad']==dif])
    print(f'  {dif}: score promedio = {score:.3f} ({n} recetas)')
print()
print('Conclusion: las recetas de dificultad Facil tienen mayor score de alacena.')
print('Esto valida que el sistema debe priorizar recetas simples para mayor aprovechamiento.')

Hallazgo clave:
  Facil: score promedio = 0.626 (19 recetas)
  Media: score promedio = 0.376 (83 recetas)
  Dificil: score promedio = 0.316 (1 recetas)

Conclusion: las recetas de dificultad Facil tienen mayor score de alacena.
Esto valida que el sistema debe priorizar recetas simples para mayor aprovechamiento.


## 6. Exportación

In [101]:
export_df = recipes_df.copy()
export_df['ingredientes'] = export_df['ingredientes'].apply(lambda x: ', '.join(x))
export_df = export_df.drop(columns=['ingredientes_texto'], errors='ignore')
export_df.to_csv('recipes_df.csv', index=False)
print(f'recipes_df.csv: {export_df.shape[0]} recetas, {export_df.shape[1]} columnas')

recipes_df.csv: 103 recetas, 21 columnas


In [102]:
scipy.sparse.save_npz('tfidf_matrix.npz', tfidf_matrix)
with open('tfidf_feature_names.json', 'w') as f:
    json.dump(list(tfidf.get_feature_names_out()), f)
print('tfidf_matrix.npz exportado')
print('tfidf_feature_names.json exportado')
print()
print('Archivos listos para:')
print('  1. Notebook de Clustering')
print('  2. Tools de la app de Streamlit')

tfidf_matrix.npz exportado
tfidf_feature_names.json exportado

Archivos listos para:
  1. Notebook de Clustering
  2. Tools de la app de Streamlit


In [103]:
# Guardar tambien el top_60 para usarlo en las tools
with open('top_ingredientes_alacena.json', 'w') as f:
    json.dump(list(top_60), f, ensure_ascii=False)
print('top_ingredientes_alacena.json exportado')
print(f'  {len(top_60)} ingredientes de alacena tipica mexicana')

top_ingredientes_alacena.json exportado
  60 ingredientes de alacena tipica mexicana
